<a href="https://colab.research.google.com/github/mithra-malicious/carisurg_portfolio/blob/main/TRIAGE_LOGIC_PSUEDO%2BPYTHON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
FUNCTION triage_emergency_patient(patient_raw_data):


    gender_clean = strip_spaces_and_lowercase(patient_raw_data.gender)
    gender_binary = map_to_binary(gender_clean)

    GCS   = convert_to_numeric_or_use_median(patient_raw_data.gcs)
    SBP   = convert_to_numeric_or_use_median(patient_raw_data.sbp)
    pulse = convert_to_numeric_or_use_median(patient_raw_data.pulse)
    Temp  = convert_to_numeric_or_use_median(patient_raw_data.temp)


    IF GCS <= 8 THEN:
        RETURN "LEVEL 1: HIGH RISK - Immediate Resuscitation (Severe Neurological Compromise)"
    IF SBP < 90 OR SBP > 200 THEN:
        RETURN "LEVEL 1: HIGH RISK - Immediate Resuscitation (Hemodynamic Instability)"
    IF pulse < 40 OR pulse > 140 THEN:
        RETURN "LEVEL 1: HIGH RISK - Immediate Resuscitation (Extreme Arrhythmia / Shock)"
    IF Temp < 35.0 OR Temp >= 40.0 THEN:
        RETURN "LEVEL 1: HIGH RISK - Immediate Resuscitation (Critical Hypothermia or Hyperpyrexia)"


    abnormal_metric_count = 0

    IF GCS >= 9 AND GCS <= 13 THEN:
        abnormal_metric_count = abnormal_metric_count + 1
    IF (SBP >= 140 AND SBP <= 200) OR (SBP >= 90 AND SBP <= 100) THEN:
        abnormal_metric_count = abnormal_metric_count + 1
    IF (pulse >= 100 AND pulse <= 140) OR (pulse >= 40 AND pulse <= 59) THEN:
        abnormal_metric_count = abnormal_metric_count + 1
    IF Temp >= 38.0 AND Temp < 40.0 THEN:
        abnormal_metric_count = abnormal_metric_count + 1

    IF abnormal_metric_count >= 2 THEN:
        RETURN "LEVEL 2: MODERATE RISK - Urgent Assessment (Multiple Compounding Abnormal Vitals)"


    RETURN "LEVEL 3: LOW RISK - Routine Standard Care (Physiologically Stable)"

# END OF MODEL

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

clinical_bounds = {
    'GCS':   {'min': 3.0,  'max': 15.0},
    'SBP':   {'min': 60.0, 'max': 250.0},
    'DBP':   {'min': 30.0, 'max': 150.0},
    'MAP':   {'min': 40.0, 'max': 180.0},
    'pulse': {'min': 30.0, 'max': 220.0},
    'Temp':  {'min': 34.0, 'max': 43.0},
    'RR':    {'min': 8.0,  'max': 60.0},
    'Fio2':  {'min': 21.0, 'max': 100.0}
}

FILE_PATH = 'EmergencyTriageDataset_Reduced_Dirty.csv'
if os.path.exists(FILE_PATH):
    df_clean = pd.read_csv(FILE_PATH)

    gender_map = {'male': 1, '1': 1, '1.0': 1, 'female': 0, '0': 0, '0.0': 0}
    df_clean['Gender_Clean'] = df_clean['Gender'].astype(str).str.lower().str.strip().map(gender_map)
    df_clean = df_clean.drop(columns=['Gender']).rename(columns={'Gender_Clean': 'Gender'})
    df_clean['Gender'] = df_clean['Gender'].fillna(df_clean['Gender'].median())


    for column, bounds in clinical_bounds.items():
        df_clean[column] = pd.to_numeric(df_clean[column], errors='coerce')
        df_clean.loc[(df_clean[column] < bounds['min']) | (df_clean[column] > bounds['max']), column] = np.nan
        df_clean[column] = df_clean[column].fillna(df_clean[column].median())

    print("Ingestion & Cleaning Engine Finalized Successfully.")
    print(df_clean.describe())
else:
    print("Execution halted: Source file path unavailable.")